# Module 1 — Generative AI Foundations, Translated for Telecom

**NetOps Co. · CELL-031A · ~30 minutes**

By the end of this notebook you will have built a working tool: an incident summarizer
that turns five raw trouble tickets into a shift-handoff briefing.

Three concepts first, quickly, because you are a technical audience and mostly half-know them:

| | |
|---|---|
| **Tokens** | Word-pieces, not words. `CELL-031A` is 3–4 tokens. Pricing and limits are measured here. |
| **Context window** | Everything the model sees at once — your prompt, your documents, its answer. |
| **Hallucination** | A fluent, confident, wrong answer. It will invent a root cause if you don't give it real data. |

Hold on to that third one. It is the thread running through the rest of the course.

## Setup — about 60 seconds

Run this once per session. Colab gives you a fresh machine each time, so the clone and
install have to happen again — that is normal, not a mistake.

**Your API key.** Click the 🔑 key icon in the left sidebar, add a secret named
`GEMINI_API_KEY`, and toggle *Notebook access* on. Get a free key at
[aistudio.google.com](https://aistudio.google.com) — no credit card.

Never paste a key into a cell. Notebooks get shared, and the key goes with them.

In [ ]:
!git clone -q https://github.com/telcobytes/netops-genai-course.git 2>/dev/null || (cd netops-genai-course && git pull -q)
!pip install -q google-genai pydantic

import sys, os
sys.path.append('/content/netops-genai-course/data')

# Key from Colab secrets, with a local fallback so this notebook also runs
# in plain Jupyter.
try:
    from google.colab import userdata
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    if not os.environ.get('GEMINI_API_KEY'):
        import getpass
        os.environ['GEMINI_API_KEY'] = getpass.getpass('GEMINI_API_KEY: ')

print('Key loaded:', bool(os.environ.get('GEMINI_API_KEY')))
print('Module 1 — Incident Summarizer')

### Sanity check — no API key needed

`mock_tools.py` is pure standard library. If this prints numbers, your environment is
working and the rest of the notebook will run.

In [ ]:
import mock_tools
summary = mock_tools.get_cell_kpis('CELL-031A')
print('window     :', summary['window_start'], '->', summary['window_end'])
print('samples    :', summary['sample_count'])
print('rolling avg:', summary['rolling_avg'])
for c in summary['thresholds_crossed']:
    print(f"  CROSSED  {c['metric']} = {c['value']} ({c['comparison']} {c['threshold']})")

---
## 1. The three-line API pattern

Talking to a model is less ceremony than people expect. Initialize a client, call it, read the text.

Watch what goes **inside** `contents`: three real numbers. That is the entire difference between
a useful diagnosis and a textbook recitation.

The one line that is *not* ceremony is `resolve_model()`. Model ids get retired while
the docs still list them as fine, so the course keeps its list in one file rather than
typing a name into each lab. You will see that file in a moment.

In [ ]:
from google import genai
from llm_client import resolve_model

client = genai.Client()   # reads GEMINI_API_KEY from the environment

# The model name is NOT typed in here. Google retires models on its own schedule
# — this course has had two die mid-lecture — so resolve_model() hands back the
# first model on the course's list that your key can actually see. One list, in
# llm_client.py, for every lab. Everything else below is the raw SDK call.
MODEL = resolve_model()
print('using model:', MODEL)

response = client.models.generate_content(
    model=MODEL,
    contents='''Cell CELL-031A at 08:45:
- PRB utilization: 91.8%
- RRC drop rate: 7.6% (threshold 5%)
- Active users: 201 (planned capacity 150)
Explain the likely immediate cause in 2 sentences.'''
)
print(response.text)

### The same question, ungrounded

Now take the numbers away and ask about a cell the model has never heard of.
Read the answer carefully before you scroll on.

In [ ]:
bad = client.models.generate_content(
    model=MODEL,          # same model as above, resolved once in the cell above
    contents='Why does CELL-031A at NetOps Co. keep congesting in the evenings?'
)
print(bad.text)

> It answered. Fluently. Possibly with a 3GPP clause. And it has never seen your network.
>
> That is hallucination — not a refusal you would catch, but confident telecom vocabulary
> wrapped around an invented fact. **Module 4 is about fixing this properly.**

---
## 2. The data you are working with

Five trouble tickets from NetOps Co.'s ticketing system. Somebody — probably you, at 2am —
has to read these and write the morning handoff.

In [ ]:
import csv, nb_viz
from IPython.display import HTML, display

with open('/content/netops-genai-course/data/tickets.csv', newline='') as f:
    tickets = list(csv.DictReader(f))

display(HTML(nb_viz.table(tickets, caption='tickets.csv — 5 rows a human has to read')))

---
## 3. Build the summarizer

Two things worth noticing in the prompt below:

1. We format the rows into **plain prose** before sending them. Models handle prose better than raw CSV.
2. The prompt states **who it is for** and **what shape** the answer takes. Specificity beats a vague ask.

> **Why this notebook writes its own `summarize()`**
>
> Every other lab imports from its module folder and re-implements nothing. This one
> is the deliberate exception. Module 1's whole point is watching the abstraction get
> built: a raw `genai` call first, then the same thing through `llm_client`, then
> wrapped in a function you can edit. So the function is written here in front of you.
>
> `summarizer.py`, next to this notebook, is the finished version of exactly this —
> same idea, tidied up, with `load_tickets()` split out. Open it after the notebook
> and the diff is the lesson.


In [ ]:
from llm_client import call_llm

def summarize(rows):
    text = '\n'.join(
        f"- [{t['ticket_id']}] {t['site_id']} ({t['category']}): {t['summary']}"
        for t in rows)
    prompt = f'''Write a NOC shift-handoff briefing from these tickets.
Group by site, flag anything urgent or recurring. Under 150 words.

TICKETS:
{text}'''
    return call_llm([{'role': 'user', 'content': prompt}])

print(summarize(tickets))

Five seconds of API time, in place of fifteen minutes of reading.

This is the **floor**. By the end of the course it will look primitive.

---
## Your turn

Change one thing at a time and compare. The point is to feel how much the prompt controls:

1. Ask for bullets instead of prose
2. Ask for CRITICAL items only
3. Ask for 40 words instead of 150

Then a harder one: remove *"group by site"* and see whether it groups anyway.

In [ ]:
# Your turn — edit the prompt and run.

def summarize_v2(rows, instruction):
    text = '\n'.join(f"- [{t['ticket_id']}] {t['site_id']}: {t['summary']}" for t in rows)
    return call_llm([{'role': 'user', 'content': f'{instruction}\n\nTICKETS:\n{text}'}])

print(summarize_v2(tickets, 'Write a NOC shift-handoff briefing. Bullets only. Under 60 words.'))

---
**Next:** Module 2 needs no code — it is the no-code baseline, and where it runs out.
Then Module 3, where we make the output something *code* can act on.